# IoT Device Identification with Adversarial Robustness
## Google Colab Pipeline (IoT IPFIX Home)

### Includes:
1. Full preprocessing & training pipeline (Steps 1-6)
2. **STEP 7** – Load saved models from Drive & CORRECTED final evaluation
3. Proper MIXED evaluation: clean + adversarial data together
4. Detector routes: adversarial → Model-2 (Robust), clean → Model-3 (Base)


In [1]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn tensorflow -q
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import os, warnings, pickle, gc
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             classification_report, accuracy_score, confusion_matrix)
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)


In [6]:
# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

DATA_PATH   = "/content/drive/MyDrive/PFE/IPFIX_ML_Instances/"
OUTPUT_PATH = "/content/drive/MyDrive/results_ml_avc/"
os.makedirs(OUTPUT_PATH, exist_ok=True)

SAMPLE_RATIO = 1.0   # <1.0 for quick testing

# SDN-compatible features (accessible through SDN controller APIs)
SDN_FEATURES = [
    "duration", "ipProto",
    "outPacketCount", "outByteCount", "inPacketCount", "inByteCount",
    "outSmallPktCount", "outLargePktCount", "outNonEmptyPktCount", "outDataByteCount",
    "outAvgIAT", "outFirstNonEmptyPktSize", "outMaxPktSize",
    "outStdevPayloadSize", "outStdevIAT", "outAvgPacketSize",
    "inSmallPktCount", "inLargePktCount", "inNonEmptyPktCount", "inDataByteCount",
    "inAvgIAT", "inFirstNonEmptyPktSize", "inMaxPktSize",
    "inStdevPayloadSize", "inStdevIAT", "inAvgPacketSize",
    "http", "https", "smb", "dns", "ntp", "tcp", "udp", "ssdp", "lan", "wan",
]

DTYPE_DICT = {
    "duration": "float32", "ipProto": "int16",
    "outPacketCount": "int32", "outByteCount": "int64",
    "inPacketCount": "int32", "inByteCount": "int64",
    "outSmallPktCount": "int32", "outLargePktCount": "int32",
    "outNonEmptyPktCount": "int32", "outDataByteCount": "int64",
    "outAvgIAT": "float32", "outFirstNonEmptyPktSize": "int32",
    "outMaxPktSize": "int32", "outStdevPayloadSize": "float32",
    "outStdevIAT": "float32", "outAvgPacketSize": "float32",
    "inSmallPktCount": "int32", "inLargePktCount": "int32",
    "inNonEmptyPktCount": "int32", "inDataByteCount": "int64",
    "inAvgIAT": "float32", "inFirstNonEmptyPktSize": "int32",
    "inMaxPktSize": "int32", "inStdevPayloadSize": "float32",
    "inStdevIAT": "float32", "inAvgPacketSize": "float32",
    "http": "int8", "https": "int8", "smb": "int8", "dns": "int8",
    "ntp": "int8", "tcp": "int8", "udp": "int8", "ssdp": "int8",
    "lan": "int8", "wan": "int8",
    "device": "category", "name": "category",
}

# 18 classes retained for IoT IPFIX Home (Table 7)
VALID_CLASSES = [
    "eclear", "sleep", "esensor", "hub-plus", "humidifier",
    "home-unit", "inkjet-printer", "smart-wifi-plug-mini", "smart-power-strip",
    "echo-dot", "fire7-tablet", "google-nest-mini", "google-chromecast",
    "atom-cam", "kasa-camera-pro", "kasa-smart-led-lamp", "fire-tv-stick-4k", "qrio-hub",
]

TARGET = "name"

# Feature categories for SDN-constraint-aware perturbation (Section 4.2)
# Independent: can be perturbed freely
# Dependent:   must change consistently with related features
# Non-modifiable: fixed network/protocol identifiers
INDEPENDENT_FEATURES = [
    "outAvgIAT", "outStdevIAT", "inAvgIAT", "inStdevIAT",
    "outAvgPacketSize", "inAvgPacketSize",
    "outFirstNonEmptyPktSize", "inFirstNonEmptyPktSize",
    "outMaxPktSize", "inMaxPktSize",
    "outStdevPayloadSize", "inStdevPayloadSize",
    "duration",
]
DEPENDENT_FEATURES = [
    "outPacketCount", "outByteCount", "inPacketCount", "inByteCount",
    "outSmallPktCount", "outLargePktCount", "outNonEmptyPktCount", "outDataByteCount",
    "inSmallPktCount", "inLargePktCount", "inNonEmptyPktCount", "inDataByteCount",
]
NON_MODIFIABLE_FEATURES = [
    "ipProto", "http", "https", "smb", "dns", "ntp", "tcp", "udp", "ssdp", "lan", "wan",
]


In [7]:
# ─────────────────────────────────────────────
# STEP 1 – DATA LOADING & PREPROCESSING
# ─────────────────────────────────────────────

def load_csv_optimized(filepath):
    available = pd.read_csv(filepath, nrows=0).columns.tolist()
    usecols = [c for c in SDN_FEATURES + [TARGET] if c in available]
    dtype_subset = {k: v for k, v in DTYPE_DICT.items() if k in usecols}
    return pd.read_csv(filepath, usecols=usecols, dtype=dtype_subset, low_memory=True)


def load_all_data(data_path=DATA_PATH, sample_ratio=1.0):
    print("=" * 60)
    print("STEP 1 – Loading IoT IPFIX Home Dataset")
    print("=" * 60)
    all_dfs = []
    for i in range(1, 13):
        fp = os.path.join(data_path, f"home{i}_labeled.csv")
        if os.path.exists(fp):
            print(f"  Loading home{i}_labeled.csv ...")
            df = load_csv_optimized(fp)
            if sample_ratio < 1.0:
                df = df.sample(frac=sample_ratio, random_state=42)
            mem = df.memory_usage(deep=True).sum() / 1024 ** 2
            print(f"    Shape: {df.shape} | Memory: {mem:.1f} MB")
            all_dfs.append(df)
            gc.collect()
    df = pd.concat(all_dfs, ignore_index=True)
    del all_dfs
    gc.collect()
    print(f"\nTotal shape: {df.shape}")
    return df


def preprocess(df):
    """Clean, filter classes, handle NaNs/Infs – returns X (float32), y (str)."""
    df = df.dropna(subset=[TARGET]).copy()
    df = df[df[TARGET].isin(VALID_CLASSES)]
    print(f"  After class filtering: {len(df)} rows, {len(VALID_CLASSES)} classes")

    valid_feats = [f for f in SDN_FEATURES if f in df.columns]

    # Remove duplicate rows (Section 5.1 – step 3)
    before = len(df)
    df = df.drop_duplicates(subset=valid_feats)
    print(f"  Duplicates removed: {before - len(df)}")

    for col in valid_feats:
        if df[col].dtype == "object":
            df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].fillna(0).astype("float32")

    df = df.replace([np.inf, -np.inf], 0)

    X = df[valid_feats].values.astype("float32")
    y = df[TARGET].values
    return X, y, valid_feats


def encode_and_scale(X, y):
    """LabelEncode y; MinMax → StandardScale X (Section 5.1 steps 1-2)."""
    le = LabelEncoder()
    y_enc = le.fit_transform(y)
    n_classes = len(le.classes_)
    print(f"  Classes ({n_classes}): {list(le.classes_)}")

    mm = MinMaxScaler()
    X_mm = mm.fit_transform(X)           # [0, 1] normalisation
    std = StandardScaler()
    X_sc = std.fit_transform(X_mm)       # zero-mean, unit-variance standardisation

    with open(os.path.join(OUTPUT_PATH, "label_encoder.pkl"), "wb") as f:
        pickle.dump(le, f)
    with open(os.path.join(OUTPUT_PATH, "scaler.pkl"), "wb") as f:
        pickle.dump({"minmax": mm, "standard": std}, f)

    return X_sc, y_enc, le, n_classes


In [ ]:

# ─────────────────────────────────────────────
# STEP 2 – BASE CLASSIFIERS (Model-3) – Table 3 / Table 6
# ─────────────────────────────────────────────

def build_base_classifiers(num_classes, n_features):
    """Returns dict of base classifiers with paper-exact hyperparameters."""
    dnn = Sequential([
        Dense(256, activation="relu", input_shape=(n_features,)),
        Dense(256, activation="relu"),
        Dense(256, activation="relu"),
        Dense(num_classes, activation="softmax"),
    ])
    dnn.compile(optimizer=Adam(0.001),
                loss="sparse_categorical_crossentropy",
                metrics=["accuracy"])

    return {
        "RF": RandomForestClassifier(
            n_estimators=300, criterion="gini", max_depth=None,
            bootstrap=True, max_features="sqrt", random_state=42, n_jobs=-1,
        ),
        "KNN": KNeighborsClassifier(
            n_neighbors=5, weights="distance", algorithm="auto", n_jobs=-1,
        ),
        "XGBoost": xgb.XGBClassifier(
            n_estimators=200, eval_metric="merror", learning_rate=0.1,
            max_depth=None, random_state=42, n_jobs=-1, tree_method="hist",
        ),
        "DNN": dnn,
    }


def train_base_classifiers(models, X_train, y_train, X_test, y_test, label_classes):
    print("\n" + "=" * 60)
    print("STEP 2 – Training Base Classifiers (Model-3)")
    print("=" * 60)
    trained, results = {}, []
    es = EarlyStopping(patience=3, restore_best_weights=True)

    for name, model in models.items():
        print(f"\n  Training {name} ...")
        if name == "DNN":
            model.fit(X_train, y_train, epochs=30, batch_size=256,
                      validation_split=0.1, callbacks=[es], verbose=0)
            model.save(os.path.join(OUTPUT_PATH, "model3_dnn.h5"))
        else:
            model.fit(X_train, y_train)
            with open(os.path.join(OUTPUT_PATH, f"model3_{name.lower()}.pkl"), "wb") as f:
                pickle.dump(model, f)
        trained[name] = model
        r = evaluate(model, X_test, y_test, name, label_classes)
        results.append(r)

    # Save summary
    df_res = pd.DataFrame([{k: v for k, v in r.items() if k != 'per_class'} for r in results])
    df_res.to_csv(os.path.join(OUTPUT_PATH, "step2_base_identification_results.csv"), index=False)

    # CORRECTION 2 : Plot Figure 5 for EACH model
    for r in results:
        plot_figure_5(r, label_classes)

    return trained


In [10]:
# ─────────────────────────────────────────────
# STEP 3 – ADVERSARIAL ATTACK GENERATION (Section 4.2)
# ─────────────────────────────────────────────

class AdversarialAttackGenerator:
    """
    Implements the paper's iterative centroid-based adversarial perturbation:
      x_adv = Projection[ x0 + c·t·mask·sign(µ_target − x0)·|Difference(µ_target, x0)| ]

    L0 minimisation  → binary masks restrict which features are perturbed
                        (only INDEPENDENT + DEPENDENT, not NON_MODIFIABLE)
    L2 minimisation  → pick from the 3 closest class centroids (K-means)
    """

    def __init__(self, feature_names):
        self.feature_names = feature_names
        self.class_centroids = {}   # {class_id: centroid_vector}
        self._build_modifiable_mask(feature_names)

    def _build_modifiable_mask(self, feat_names):
        """Binary mask: 1 = modifiable (independent/dependent), 0 = locked."""
        modifiable = set(INDEPENDENT_FEATURES + DEPENDENT_FEATURES)
        self.mask = np.array(
            [1.0 if f in modifiable else 0.0 for f in feat_names], dtype=np.float32
        )

    def fit_centroids(self, X_train, y_train):
        """Compute per-class mean (centroid) vectors."""
        classes = np.unique(y_train)
        for c in classes:
            self.class_centroids[c] = X_train[y_train == c].mean(axis=0)

    def _three_closest_targets(self, x, true_class):
        """Return the 3 class IDs closest (L2) to x, excluding the true class."""
        dists = {
            c: np.linalg.norm(x - mu)
            for c, mu in self.class_centroids.items()
            if c != true_class
        }
        return sorted(dists, key=dists.get)[:3]

    def _projection(self, x_adv):
        """Clip to valid scaled range and enforce integer rounding for discrete features."""
        return np.clip(x_adv, -3.0, 3.0)

    def generate(self, x, true_class, max_iter=50, c=0.05):
        """
        Generate one adversarial example against the class centroid.
        Tries the 3 closest target classes and returns the best (lowest L2 dist).
        """
        candidates = self._three_closest_targets(x, true_class)
        best_adv, best_dist = x.copy(), np.inf

        for target_class in candidates:
            mu_t = self.class_centroids[target_class]
            x_adv = x.copy()
            for t in range(1, max_iter + 1):
                diff = mu_t - x_adv
                direction = np.sign(diff) * np.abs(diff)
                x_adv = x_adv + c * t * self.mask * direction
                x_adv = self._projection(x_adv)

            dist = np.linalg.norm(x_adv - x)
            if dist < best_dist:
                best_dist = dist
                best_adv = x_adv.copy()

        return best_adv

    def generate_batch(self, X, y, n_samples=None):
        if n_samples is None:
            n_samples = len(X)
        n_samples = min(n_samples, len(X))
        X_adv = []
        for i in range(n_samples):
            x_adv = self.generate(X[i], y[i])
            X_adv.append(x_adv)
            if (i + 1) % 500 == 0:
                print(f"    Adversarial samples generated: {i + 1}/{n_samples}")
        return np.array(X_adv, dtype=np.float32)


def generate_adversarial_samples(base_models, X_train, y_train, X_test, y_test,
                                 feature_names, label_classes, n_adv=5000):
    """
    CORRECTION 3 – Séparation train/test pour éviter le Data Leakage :
      - X_adv_train : générés depuis X_train (pour entraîner les modèles robustes à l'étape 5)
      - X_adv_test  : générés depuis X_test  (pour l'évaluation uniquement, étapes 3 et 6)
    """
    print("\n" + "=" * 60)
    print("STEP 3 – Adversarial Attack Generation (Section 4.2)")
    print("=" * 60)

    # Fit centroid generator on training data
    atk = AdversarialAttackGenerator(feature_names)
    atk.fit_centroids(X_train, y_train)

    # ── Adversarial TEST samples (pour mesurer l'impact & l'évaluation finale) ──
    X_test_sample = X_test[:n_adv]
    y_test_sample = y_test[:n_adv]
    print(f"\n  Generating {n_adv} adversarial TEST samples (from X_test) ...")
    X_adv_test = atk.generate_batch(X_test_sample, y_test_sample)

    # ── Adversarial TRAIN samples (pour entraîner les modèles robustes – CORRECTION 3) ──
    n_adv_train = min(n_adv, int(len(X_train) * 0.30))  # 30 % de X_train
    idx_train = np.random.choice(len(X_train), n_adv_train, replace=False)
    X_train_sub = X_train[idx_train]
    y_train_sub = y_train[idx_train]
    print(f"  Generating {n_adv_train} adversarial TRAIN samples (from X_train, 30%) ...")
    X_adv_train = atk.generate_batch(X_train_sub, y_train_sub)

    # ── Measure adversarial impact on all 4 base models (on test set only) ──
    print("\n  Adversarial Impact on Base Models:")
    impact_results = []

    # CORRECTION 1 & 2 : métriques complètes + Figure 7 pour CHAQUE modèle
    per_class_results = {}   # {model_name: {"clean": {cls: f1}, "adv": {cls: f1}}}

    for name, model in base_models.items():
        # Clean predictions
        y_pred_c = _predict(model, X_test_sample)
        acc_clean = accuracy_score(y_test_sample, y_pred_c)
        f1_clean  = f1_score(y_test_sample, y_pred_c, average="weighted", zero_division=0)
        pr_clean  = precision_score(y_test_sample, y_pred_c, average="weighted", zero_division=0)
        rc_clean  = recall_score(y_test_sample, y_pred_c, average="weighted", zero_division=0)

        # Adversarial predictions
        y_pred_a = _predict(model, X_adv_test)
        acc_adv  = accuracy_score(y_test_sample, y_pred_a)
        f1_adv   = f1_score(y_test_sample, y_pred_a, average="weighted", zero_division=0)
        pr_adv   = precision_score(y_test_sample, y_pred_a, average="weighted", zero_division=0)
        rc_adv   = recall_score(y_test_sample, y_pred_a, average="weighted", zero_division=0)

        drop = f1_clean - f1_adv
        print(f"    {name:10s} | Clean: Acc={acc_clean:.4f} Pr={pr_clean:.4f} Rc={rc_clean:.4f} F1={f1_clean:.4f} | "
              f"Adv: Acc={acc_adv:.4f} Pr={pr_adv:.4f} Rc={rc_adv:.4f} F1={f1_adv:.4f} | Drop={drop:.4f}")

        cr_adv = classification_report(y_test_sample, y_pred_a, target_names=label_classes, zero_division=0)
        print(f"\n  Classification Report under Attack for {name}:")
        print(cr_adv)

        with open(os.path.join(OUTPUT_PATH, f"classification_report_adv_{name}.txt"), "w", encoding="utf-8") as f:
            f.write(f"[Classification Report under Attack for {name}]\n")
            f.write(cr_adv)

        cm_adv = confusion_matrix(y_test_sample, y_pred_a)
        plt.figure(figsize=(12, 10))
        sns.heatmap(cm_adv, annot=True, fmt='d', cmap='Reds', xticklabels=label_classes, yticklabels=label_classes)
        plt.title(f'Adversarial Confusion Matrix - {name}')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_PATH, f"confusion_matrix_adv_{name}.png"))
        plt.close()

        # Per-class F1 for Figure 7 (CORRECTION 2 : pour tous les modèles)
        rep_c = classification_report(y_test_sample, y_pred_c, output_dict=True, zero_division=0)
        rep_a = classification_report(y_test_sample, y_pred_a, output_dict=True, zero_division=0)
        pc_clean, pc_adv = {}, {}
        for i, cls in enumerate(label_classes):
            idx = str(i)
            pc_clean[cls] = rep_c[idx]['f1-score'] if idx in rep_c else 0.0
            pc_adv[cls]   = rep_a[idx]['f1-score'] if idx in rep_a else 0.0
        per_class_results[name] = {"clean": pc_clean, "adv": pc_adv}

        impact_results.append({
            "model": name,
            "accuracy_clean": acc_clean, "precision_clean": pr_clean,
            "recall_clean": rc_clean,    "f1_clean": f1_clean,
            "accuracy_adv":  acc_adv,   "precision_adv": pr_adv,
            "recall_adv":    rc_adv,     "f1_adv": f1_adv,
            "drop": drop,
        })

    pd.DataFrame(impact_results).to_csv(
        os.path.join(OUTPUT_PATH, "step3_adversarial_impact.csv"), index=False
    )

    # CORRECTION 2 : Figure 7 pour chaque modèle
    for name, pc in per_class_results.items():
        plot_figure_7(pc["clean"], pc["adv"], label_classes, model_name=name)

    return X_adv_test, X_adv_train, y_train_sub, X_test_sample, y_test_sample, atk


In [ ]:
# STEP 4 – ADVERSARIAL DETECTOR – Model-1 (Table 4)
# ─────────────────────────────────────────────

def build_detector_classifiers(n_features):
    """Binary classifiers: benign (0) vs adversarial (1). Table 4 hyperparameters."""
    dnn = Sequential([
        Dense(256, activation="relu", input_shape=(n_features,)),
        Dense(256, activation="relu"),
        Dense(256, activation="relu"),
        Dense(1, activation="sigmoid"),
    ])
    dnn.compile(optimizer=Adam(0.001),
                loss="binary_crossentropy",
                metrics=["accuracy"])

    return {
        "RF": RandomForestClassifier(
            n_estimators=300, max_depth=None, max_features="sqrt",
            min_samples_split=5, random_state=42, n_jobs=-1,
        ),
        "KNN": KNeighborsClassifier(
            n_neighbors=5, weights="uniform", algorithm="brute", n_jobs=-1,
        ),
        "XGBoost": xgb.XGBClassifier(
            n_estimators=300, max_depth=3, learning_rate=0.2,
            random_state=42, n_jobs=-1, tree_method="hist",
        ),
        "DNN": dnn,
    }


def train_detectors(X_train, y_train_device,  # clean training data
                    X_adv, X_sample_clean,     # adversarial & matched clean test samples
                    n_features):
    print("\n" + "=" * 60)
    print("STEP 4 – Training Adversarial Detectors (Model-1, Table 4)")
    print("=" * 60)

    # Build binary detection dataset: clean=0, adversarial=1
    n_det = min(len(X_adv), len(X_train))
    X_det = np.vstack([X_train[:n_det], X_adv])
    y_det = np.hstack([np.zeros(n_det, dtype=np.int8), np.ones(len(X_adv), dtype=np.int8)])

    X_det_tr, X_det_ts, y_det_tr, y_det_ts = train_test_split(
        X_det, y_det, test_size=0.25, random_state=42, stratify=y_det
    )
    print(f"  Detection dataset – train: {len(X_det_tr)}  test: {len(X_det_ts)}")

    detector_models = build_detector_classifiers(n_features)
    trained_detectors = {}
    det_results = []
    es = EarlyStopping(patience=3, restore_best_weights=True)

    for name, model in detector_models.items():
        print(f"\n  Training Detector {name} ...")
        if name == "DNN":
            model.fit(X_det_tr, y_det_tr, epochs=30, batch_size=256,
                      validation_split=0.1, callbacks=[es], verbose=0)
            y_pred = (model.predict(X_det_ts) > 0.5).astype(int).flatten()
            model.save(os.path.join(OUTPUT_PATH, "model1_dnn_detector.h5"))
        else:
            model.fit(X_det_tr, y_det_tr)
            y_pred = model.predict(X_det_ts)
            with open(os.path.join(OUTPUT_PATH, f"model1_{name.lower()}_detector.pkl"), "wb") as f:
                pickle.dump(model, f)

        acc = accuracy_score(y_det_ts, y_pred)
        f1 = f1_score(y_det_ts, y_pred, average="binary", zero_division=0)
        pr = precision_score(y_det_ts, y_pred, average="binary", zero_division=0)
        rc = recall_score(y_det_ts, y_pred, average="binary", zero_division=0)
        print(f"    {name:10s} | Acc={acc:.4f} F1={f1:.4f} Precision={pr:.4f} Recall={rc:.4f}")
        trained_detectors[name] = model
        det_results.append({"model": name, "accuracy": acc, "f1": f1, "precision": pr, "recall": rc})

    pd.DataFrame(det_results).to_csv(
        os.path.join(OUTPUT_PATH, "step4_detector_results.csv"), index=False
    )

    # Plot Figure 9
    plot_figure_9(det_results)

    return trained_detectors


In [ ]:
# ─────────────────────────────────────────────
# STEP 5 – ADVERSARIAL TRAINING – Model-2 (Table 5)
# ─────────────────────────────────────────────

def build_robust_classifiers(num_classes, n_features):
    """Classifiers retrained on clean+adversarial data. Table 5 hyperparameters."""
    dnn = Sequential([
        Dense(256, activation="relu", input_shape=(n_features,)),
        Dense(256, activation="relu"),
        Dense(256, activation="relu"),
        Dense(num_classes, activation="softmax"),
    ])
    dnn.compile(optimizer=Adam(0.001),
                loss="sparse_categorical_crossentropy",
                metrics=["accuracy"])

    return {
        "RF": RandomForestClassifier(
            n_estimators=200, max_depth=None, max_features="sqrt",
            bootstrap=True, random_state=42, n_jobs=-1,
        ),
        "KNN": KNeighborsClassifier(
            n_neighbors=3, weights="distance", algorithm="auto", n_jobs=-1,
        ),
        "XGBoost": xgb.XGBClassifier(
            n_estimators=100, max_depth=7, learning_rate=0.2,
            random_state=42, n_jobs=-1, tree_method="hist",
        ),
        "DNN": dnn,
    }


def train_robust_classifiers(base_models, X_train, y_train,
                              X_adv_train, y_adv_train_true,
                              X_adv_test, X_test_sample, y_test_sample,
                              num_classes, n_features):
    """
    CORRECTION 3 – Entraîne les modèles robustes UNIQUEMENT avec X_adv_train
    (généré depuis X_train), jamais depuis X_test → pas de Data Leakage.
    L'évaluation utilise X_adv_test (généré depuis X_test).
    """
    print("\n" + "=" * 60)
    print("STEP 5 – Adversarial Training / Robust Classifiers (Model-2, Table 5)")
    print("=" * 60)

    # Joint corpus: clean training data + adversarial TRAIN samples
    X_robust = np.vstack([X_train, X_adv_train])
    y_robust  = np.hstack([y_train, y_adv_train_true])
    print(f"  Robust training set size: {len(X_robust)} samples "
          f"(X_train={len(X_train)} + X_adv_train={len(X_adv_train)})")

    robust_models = build_robust_classifiers(num_classes, n_features)
    trained_robust = {}
    rob_results = []
    es = EarlyStopping(patience=3, restore_best_weights=True)

    for name, model in robust_models.items():
        print(f"\n  Training Robust {name} ...")
        if name == "DNN":
            model.fit(X_robust, y_robust, epochs=30, batch_size=256,
                      validation_split=0.1, callbacks=[es], verbose=0)
            model.save(os.path.join(OUTPUT_PATH, "model2_dnn_robust.h5"))
        else:
            model.fit(X_robust, y_robust)
            with open(os.path.join(OUTPUT_PATH, f"model2_{name.lower()}_robust.pkl"), "wb") as f:
                pickle.dump(model, f)

        trained_robust[name] = model

        # CORRECTION 1 : toutes les métriques (Accuracy, Precision, Recall, F1)
        y_pred_clean = _predict(model, X_test_sample)
        y_pred_adv   = _predict(model, X_adv_test)
        y_pred_base  = _predict(base_models[name], X_adv_test)

        acc_clean = accuracy_score(y_test_sample, y_pred_clean)
        f1_clean  = f1_score(y_test_sample, y_pred_clean, average="weighted", zero_division=0)
        pr_clean  = precision_score(y_test_sample, y_pred_clean, average="weighted", zero_division=0)
        rc_clean  = recall_score(y_test_sample, y_pred_clean, average="weighted", zero_division=0)

        acc_adv = accuracy_score(y_test_sample, y_pred_adv)
        f1_adv  = f1_score(y_test_sample, y_pred_adv, average="weighted", zero_division=0)
        pr_adv  = precision_score(y_test_sample, y_pred_adv, average="weighted", zero_division=0)
        rc_adv  = recall_score(y_test_sample, y_pred_adv, average="weighted", zero_division=0)

        f1_base  = f1_score(y_test_sample, y_pred_base, average="weighted", zero_division=0)
        recovery = (f1_adv / f1_clean * 100) if f1_clean > 0 else 0

        print(f"    {name:10s} | Clean: Acc={acc_clean:.4f} Pr={pr_clean:.4f} Rc={rc_clean:.4f} F1={f1_clean:.4f} | "
              f"Adv Base: F1={f1_base:.4f} | Adv Robust: Acc={acc_adv:.4f} Pr={pr_adv:.4f} Rc={rc_adv:.4f} F1={f1_adv:.4f} | Recovery={recovery:.1f}%")
        rob_results.append({
            "model": name,
            "accuracy_clean": acc_clean, "precision_clean": pr_clean,
            "recall_clean":   rc_clean,   "f1_clean":        f1_clean,
            "accuracy_adv":   acc_adv,    "precision_adv":   pr_adv,
            "recall_adv":     rc_adv,     "f1_adv_after_robust": f1_adv,
            "f1_adv_before_robust": f1_base,
            "recovery_pct": recovery,
        })

    pd.DataFrame(rob_results).to_csv(
        os.path.join(OUTPUT_PATH, "step5_robust_training_results.csv"), index=False
    )
    return trained_robust


In [ ]:
# STEP 6 – TWO-TIERED DEFENSE EVALUATION (Figure 4)
# ─────────────────────────────────────────────

def two_tiered_defense_evaluation(base_models, detector_models, robust_models,
                                   X_test_clean, y_test_clean,
                                   X_adv_test, y_adv_true):
    """
    Pipeline (Figure 4):
      Incoming flow → Model-1 (Detector)
        → Flagged as adversarial → classify with Model-2 (Robust classifier)
        → Flagged as benign     → classify with Model-3 (Base classifier)

    CORRECTION 3 : X_adv_test provient uniquement de X_test (pas de Data Leakage).
    CORRECTION 1 : métriques complètes (Accuracy, Precision, Recall, F1) dans les résultats.
    """
    print("\n" + "=" * 60)
    print("STEP 6 – Two-Tiered Defense Evaluation (Figure 4)")
    print("=" * 60)

    pipeline_results = []

    for det_name, detector in detector_models.items():
        print(f"\n  Detector: {det_name}")

        # ── Evaluate on CLEAN test samples ──
        if det_name == "DNN":
            det_clean = (detector.predict(X_test_clean) > 0.5).astype(int).flatten()
        else:
            det_clean = detector.predict(X_test_clean)

        # ── Evaluate on ADVERSARIAL test samples (CORRECTION 3 : X_adv_test isolé) ──
        if det_name == "DNN":
            det_adv = (detector.predict(X_adv_test) > 0.5).astype(int).flatten()
        else:
            det_adv = detector.predict(X_adv_test)

        for cls_name in base_models:
            base_model   = base_models[cls_name]
            robust_model = robust_models[cls_name]

            # ── Clean samples through pipeline ──
            y_pred_clean = np.where(
                det_clean == 0,
                _predict(base_model, X_test_clean),      # benign → Model-3
                _predict(robust_model, X_test_clean),    # flagged → Model-2
            )
            # CORRECTION 1 : toutes les métriques sur les données clean
            acc_clean = accuracy_score(y_test_clean, y_pred_clean)
            f1_clean  = f1_score(y_test_clean, y_pred_clean, average="weighted", zero_division=0)
            pr_clean  = precision_score(y_test_clean, y_pred_clean, average="weighted", zero_division=0)
            rc_clean  = recall_score(y_test_clean, y_pred_clean, average="weighted", zero_division=0)

            # ── Adversarial samples through pipeline ──
            y_pred_adv = np.where(
                det_adv == 0,
                _predict(base_model, X_adv_test),        # not detected → Model-3
                _predict(robust_model, X_adv_test),      # detected     → Model-2
            )
            # CORRECTION 1 : toutes les métriques sur les données adversariales
            acc_adv = accuracy_score(y_adv_true, y_pred_adv)
            f1_adv  = f1_score(y_adv_true, y_pred_adv, average="weighted", zero_division=0)
            pr_adv  = precision_score(y_adv_true, y_pred_adv, average="weighted", zero_division=0)
            rc_adv  = recall_score(y_adv_true, y_pred_adv, average="weighted", zero_division=0)

            # Baseline : Metrics without defense
            y_pred_no_def = _predict(base_model, X_adv_test)
            acc_no_defense = accuracy_score(y_adv_true, y_pred_no_def)
            pr_no_defense  = precision_score(y_adv_true, y_pred_no_def, average="weighted", zero_division=0)
            rc_no_defense  = recall_score(y_adv_true, y_pred_no_def, average="weighted", zero_division=0)
            f1_no_defense  = f1_score(y_adv_true, y_pred_no_def, average="weighted", zero_division=0)

            recovery = (f1_adv / f1_no_defense * 100) if f1_no_defense > 0 else 0

            print(f"    Classifier={cls_name:10s}\n"
                  f"      Clean      | Acc={acc_clean:.4f} Pr={pr_clean:.4f} Rc={rc_clean:.4f} F1={f1_clean:.4f}\n"
                  f"      Adv NoDef  | Acc={acc_no_defense:.4f} Pr={pr_no_defense:.4f} Rc={rc_no_defense:.4f} F1={f1_no_defense:.4f}\n"
                  f"      Adv Defend | Acc={acc_adv:.4f} Pr={pr_adv:.4f} Rc={rc_adv:.4f} F1={f1_adv:.4f}\n"
                  f"      Recovery   | {recovery:.1f}%")

            pipeline_results.append({
                "detector": det_name, "classifier": cls_name,
                "accuracy_clean": acc_clean, "precision_clean": pr_clean,
                "recall_clean":   rc_clean,   "f1_clean":        f1_clean,
                "accuracy_adv":   acc_adv,    "precision_adv":   pr_adv,
                "recall_adv":     rc_adv,
                "accuracy_adv_no_defense": acc_no_defense,
                "precision_adv_no_defense": pr_no_defense,
                "recall_adv_no_defense": rc_no_defense,
                "f1_adv_no_defense": f1_no_defense,
                "f1_adv_defended":   f1_adv,
                "recovery_pct":      recovery,
            })

    df_res = pd.DataFrame(pipeline_results)
    df_res.to_csv(os.path.join(OUTPUT_PATH, "step6_twotiered_defense_results.csv"), index=False)
    print(f"\n  Results saved to {OUTPUT_PATH}")

    # Plot Figure 10
    plot_figure_10(df_res)

    return df_res


In [ ]:
# UTILITIES
# ─────────────────────────────────────────────

def _predict(model, X):
    """Unified predict that handles Keras (softmax) and sklearn models."""
    y_pred = model.predict(X)
    if hasattr(y_pred, "shape") and len(y_pred.shape) > 1 and y_pred.shape[1] > 1:
        return np.argmax(y_pred, axis=1)
    if len(y_pred.shape) == 2 and y_pred.shape[1] == 1:
        return (y_pred > 0.5).astype(int).flatten()
    return y_pred.flatten().astype(int)


def evaluate(model, X_test, y_test, name, label_classes=None):
    y_pred = _predict(model, X_test)
    f1  = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    pr  = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rc  = recall_score(y_test, y_pred, average="weighted", zero_division=0)

    per_class = {}
    if label_classes is not None:
        cr_str = classification_report(y_test, y_pred, target_names=label_classes, zero_division=0)
        print(f"\n[Classification Report for {name}]")
        print(cr_str)

        with open(os.path.join(OUTPUT_PATH, f"classification_report_{name}.txt"), "w", encoding="utf-8") as f:
            f.write(f"[Classification Report for {name}]\n")
            f.write(cr_str)

        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(12, 10))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_classes, yticklabels=label_classes)
        plt.title(f'Confusion Matrix - {name}')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_PATH, f"confusion_matrix_{name}.png"))
        plt.close()

        report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
        for i, cls in enumerate(label_classes):
            idx = str(i)
            if idx in report:
                per_class[cls] = {"precision": report[idx]["precision"], "recall": report[idx]["recall"], "f1": report[idx]["f1-score"]}
            else:
                per_class[cls] = {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    acc = accuracy_score(y_test, y_pred)
    print(f"    {name:10s} | Acc={acc:.4f} F1={f1:.4f} Precision={pr:.4f} Recall={rc:.4f}")
    return {"model": name, "accuracy": acc, "f1": f1, "precision": pr, "recall": rc, "per_class": per_class}


In [ ]:
# PLOTTING UTILITIES FOR PAPER FIGURES
# ─────────────────────────────────────────────

def plot_figure_5(result, label_classes):
    """
    Figure 5: IoT Device Identification Scores in IoT IPFIX Home.
    Individual plot per model with paper-like annotations.
    """
    name = result.get("model", "Unknown")
    metrics = result.get("per_class", {})
    if not metrics:
        return

    df_plot = pd.DataFrame(metrics).T
    # Make sure order matches label_classes
    df_plot = df_plot.reindex(label_classes).fillna(0)

    fig, ax = plt.subplots(figsize=(16, 6))

    x = np.arange(len(label_classes))
    width = 0.25

    bars1 = ax.bar(x - width, df_plot["precision"] * 100, width, label='Precision')
    bars2 = ax.bar(x,          df_plot["recall"]    * 100, width, label='Recall')
    bars3 = ax.bar(x + width,  df_plot["f1"]        * 100, width, label='F1-Score')

    # Add text labels above bars as seen in the specified image
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width() / 2.,
                        height + 1,
                        f"{height:.1f}%",
                        ha='center', va='bottom', rotation=90, fontsize=8)

    ax.set_ylim(40, 110) # Set reasonable y limit to leave space for labels
    ax.set_ylabel('Score (%)')
    ax.set_title(name, pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(label_classes, rotation=45, ha="right")

    # remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # horizontal grid lines
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    ax.legend(loc='upper left', bbox_to_anchor=(0.0, 1.15), ncol=3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, f'Figure_5_Device_Identification_{name}.png'))
    plt.close()


def plot_figure_7(clean_f1, adv_f1, label_classes, model_name=""):
    """
    Figure 7: Adversarial effect on IoT IPFIX Home identification device.
    CORRECTION 2 : génère une image par modèle (model_name).
    """
    plt.figure(figsize=(14, 6))
    x = np.arange(len(label_classes))
    width = 0.35

    c_f1 = [clean_f1.get(cls, 0.0) * 100 for cls in label_classes]
    a_f1 = [adv_f1.get(cls, 0.0)   * 100 for cls in label_classes]

    plt.bar(x - width / 2, c_f1, width, label='Clean F1-Score',       color='blue')
    plt.bar(x + width / 2, a_f1, width, label='Adversarial F1-Score', color='red')

    plt.xlabel('IoT Devices')
    plt.ylabel('F1-Score (%)')
    suffix = f" ({model_name})" if model_name else ""
    plt.title(f'Figure 7: Adversarial effect on IoT IPFIX Home identification device{suffix}')
    plt.xticks(x, label_classes, rotation=45, ha="right")
    plt.legend()
    plt.tight_layout()
    fname = f'Figure_7_Adversarial_Effect_{model_name}.png' if model_name else 'Figure_7_Adversarial_Effect.png'
    plt.savefig(os.path.join(OUTPUT_PATH, fname))
    plt.close()

def plot_figure_9(det_results):
    """Figure 9: Adversarial Instance Detection."""
    df_plot = pd.DataFrame(det_results)

    plt.figure(figsize=(10, 6))
    x = np.arange(len(df_plot))
    width = 0.25

    plt.bar(x - width, df_plot["precision"]*100, width, label='Precision')
    plt.bar(x, df_plot["recall"]*100, width, label='Recall')
    plt.bar(x + width, df_plot["f1"]*100, width, label='F1-Score')

    plt.xlabel('Detection Models')
    plt.ylabel('Score (%)')
    plt.title('Figure 9: Adversarial Instance Detection')
    plt.xticks(x, df_plot["model"])
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'Figure_9_Adversarial_Detection.png'))
    plt.close()

def plot_figure_10(df_res):
    """Figure 10: Performance Evaluation of Device Identification With Robustness Measures."""
    # Pick the best detector, typically DNN, for the plot
    dnn_res = df_res[df_res['detector'] == 'DNN']
    if dnn_res.empty:
        dnn_res = df_res

    models = dnn_res['classifier'].tolist()
    f1_clean = dnn_res['f1_clean'] * 100
    f1_adv_nodef = dnn_res['f1_adv_no_defense'] * 100
    f1_adv_def = dnn_res['f1_adv_defended'] * 100

    plt.figure(figsize=(12, 6))
    x = np.arange(len(models))
    width = 0.25

    plt.bar(x - width, f1_clean, width, label='Clean F1-Score', color='green')
    plt.bar(x, f1_adv_nodef, width, label='F1-Score under Attack', color='red')
    plt.bar(x + width, f1_adv_def, width, label='F1-Score with Robustness', color='blue')

    plt.xlabel('Classification Models')
    plt.ylabel('F1-Score (%)')
    plt.title('Figure 10: Evaluation in IoT IPFIX Home With Robustness Measures')
    plt.xticks(x, models)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'Figure_10_Robustness_Measures.png'))
    plt.close()



In [ ]:
# MAIN ENTRY POINT
# ─────────────────────────────────────────────

def main():
    # ── Step 1: Load & Preprocess ──────────────────────────────
    df = load_all_data(DATA_PATH, sample_ratio=SAMPLE_RATIO)
    X, y, feature_names = preprocess(df)
    del df; gc.collect()

    X_sc, y_enc, label_encoder, num_classes = encode_and_scale(X, y)
    del X; gc.collect()

    # 75/25 split (Section 5.1)
    X_train, X_test, y_train, y_test = train_test_split(
        X_sc, y_enc, test_size=0.25, random_state=42, stratify=y_enc
    )
    print(f"\n  Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")
    n_features = X_train.shape[1]
    label_classes = list(label_encoder.classes_)

    # ── Step 2: Base Classifiers (Model-3) ─────────────────────
    base_models_cfg = build_base_classifiers(num_classes, n_features)
    base_models     = train_base_classifiers(
        base_models_cfg, X_train, y_train, X_test, y_test, label_classes
    )

    # ── Step 3: Adversarial Attack Generation ──────────────────
    # CORRECTION 3 : génère X_adv_test (eval) ET X_adv_train (robustness training)
    N_ADV = min(5000, len(X_test) // 4)
    X_adv_test, X_adv_train, y_adv_train_true, X_test_sample, y_test_sample, attacker = (
        generate_adversarial_samples(
            base_models, X_train, y_train, X_test, y_test,
            feature_names, label_classes, n_adv=N_ADV
        )
    )

    # ── Step 4: Train Detectors (Model-1, Table 4) ─────────────
    # Le détecteur utilise X_adv_test pour construire son jeu de détection
    detector_models = train_detectors(
        X_train, y_train, X_adv_test, X_test_sample, n_features
    )

    # ── Step 5: Adversarial Training (Model-2, Table 5) ────────
    # CORRECTION 3 : entraîné avec X_adv_train (depuis X_train), évalué sur X_adv_test
    robust_models = train_robust_classifiers(
        base_models, X_train, y_train,
        X_adv_train, y_adv_train_true,
        X_adv_test, X_test_sample, y_test_sample,
        num_classes, n_features
    )

    # ── Step 6: Two-Tiered Defense Evaluation (Figure 4) ───────
    # CORRECTION 3 : X_test_sample (clean) vs X_adv_test (adversarial)
    two_tiered_defense_evaluation(
        base_models, detector_models, robust_models,
        X_test_sample, y_test_sample, X_adv_test, y_test_sample
    )

    print("\n" + "=" * 60)
    print("PIPELINE COMPLETE – All results saved to:", OUTPUT_PATH)
    print("=" * 60)


if __name__ == "__main__":
    main()



In [ ]:
# Uncomment to run full training pipeline
# main()


---
# STEP 7 – Load Saved Models & Corrected Final Evaluation

## Logique corrigée du Two-Tiered Defense (Figure 4 du paper) :
1. On prépare un jeu de test **MIXTE** : données clean + données adversariales
2. Le **Détecteur (Model-1)** prédit pour chaque échantillon : clean (0) ou adversarial (1)
3. Si **détecté adversarial** → envoyé au **Model-2 (Robust classifier)**
4. Si **détecté clean** → envoyé au **Model-3 (Base classifier)**
5. Métriques calculées sur le jeu **MIXTE complet**

### Corrections par rapport au code précédent :
- ❌ **Avant** : clean et adversarial évalués séparément → métriques biaisées
- ✅ **Maintenant** : jeu MIXTE (comme dans le paper), recovery = F1_defended / F1_clean
- ❌ **Avant** : test sur seulement `X_test[:n_adv]` (sous-ensemble)
- ✅ **Maintenant** : test sur tout le jeu de test


In [4]:
# ═══════════════════════════════════════════════════
# STEP 7A – Load all saved models from Google Drive
# ═══════════════════════════════════════════════════

MODEL_PATH = '/content/drive/MyDrive/results_ml_avc/'

def load_sklearn_model(filepath):
    with open(filepath, 'rb') as f:
        return pickle.load(f)

# ── Load Model-3 (Base classifiers) ──
print('Loading Base Classifiers (Model-3)...')
base_models = {}
for name, fname in [('RF', 'model3_rf.pkl'), ('KNN', 'model3_knn.pkl'),
                     ('XGBoost', 'model3_xgboost.pkl')]:
    fpath = os.path.join(MODEL_PATH, fname)
    if os.path.exists(fpath):
        base_models[name] = load_sklearn_model(fpath)
        print(f'  ✓ {name} loaded from {fname}')
    else:
        print(f'  ✗ {name} NOT FOUND: {fpath}')

dnn_path = os.path.join(MODEL_PATH, 'model3_dnn.h5')
if os.path.exists(dnn_path):
    base_models['DNN'] = load_model(dnn_path)
    print(f'  ✓ DNN loaded from model3_dnn.h5')
else:
    print(f'  ✗ DNN NOT FOUND: {dnn_path}')

# ── Load Model-1 (Detectors) ──
print('\nLoading Detectors (Model-1)...')
detector_models = {}
for name, fname in [('RF', 'model1_rf_detector.pkl'), ('KNN', 'model1_knn_detector.pkl'),
                     ('XGBoost', 'model1_xgboost_detector.pkl')]:
    fpath = os.path.join(MODEL_PATH, fname)
    if os.path.exists(fpath):
        detector_models[name] = load_sklearn_model(fpath)
        print(f'  ✓ {name} loaded from {fname}')
    else:
        print(f'  ✗ {name} NOT FOUND: {fpath}')

det_dnn_path = os.path.join(MODEL_PATH, 'model1_dnn_detector.h5')
if os.path.exists(det_dnn_path):
    detector_models['DNN'] = load_model(det_dnn_path)
    print(f'  ✓ DNN loaded from model1_dnn_detector.h5')
else:
    print(f'  ✗ DNN NOT FOUND: {det_dnn_path}')

# ── Load Model-2 (Robust classifiers) ──
print('\nLoading Robust Classifiers (Model-2)...')
robust_models = {}
for name, fname in [('RF', 'model2_rf_robust.pkl'), ('KNN', 'model2_knn_robust.pkl'),
                     ('XGBoost', 'model2_xgboost_robust.pkl')]:
    fpath = os.path.join(MODEL_PATH, fname)
    if os.path.exists(fpath):
        robust_models[name] = load_sklearn_model(fpath)
        print(f'  ✓ {name} loaded from {fname}')
    else:
        print(f'  ✗ {name} NOT FOUND: {fpath}')

rob_dnn_path = os.path.join(MODEL_PATH, 'model2_dnn_robust.h5')
if os.path.exists(rob_dnn_path):
    robust_models['DNN'] = load_model(rob_dnn_path)
    print(f'  ✓ DNN loaded from model2_dnn_robust.h5')
else:
    print(f'  ✗ DNN NOT FOUND: {rob_dnn_path}')

# ── Load label encoder & scaler ──
print('\nLoading Label Encoder & Scaler...')
with open(os.path.join(MODEL_PATH, 'label_encoder.pkl'), 'rb') as f:
    label_encoder = pickle.load(f)
with open(os.path.join(MODEL_PATH, 'scaler.pkl'), 'rb') as f:
    scalers = pickle.load(f)

label_classes = list(label_encoder.classes_)
num_classes = len(label_classes)
print(f'  Classes ({num_classes}): {label_classes}')
print(f'\n✅ All models loaded successfully!')


Loading Base Classifiers (Model-3)...
  ✓ RF loaded from model3_rf.pkl
  ✓ KNN loaded from model3_knn.pkl
  ✓ XGBoost loaded from model3_xgboost.pkl


  ✓ DNN loaded from model3_dnn.h5

Loading Detectors (Model-1)...
  ✓ RF loaded from model1_rf_detector.pkl
  ✓ KNN loaded from model1_knn_detector.pkl
  ✓ XGBoost loaded from model1_xgboost_detector.pkl


  ✓ DNN loaded from model1_dnn_detector.h5

Loading Robust Classifiers (Model-2)...
  ✓ RF loaded from model2_rf_robust.pkl
  ✓ KNN loaded from model2_knn_robust.pkl
  ✓ XGBoost loaded from model2_xgboost_robust.pkl


  ✓ DNN loaded from model2_dnn_robust.h5

Loading Label Encoder & Scaler...
  Classes (18): ['atom-cam', 'echo-dot', 'eclear', 'esensor', 'fire-tv-stick-4k', 'fire7-tablet', 'google-chromecast', 'google-nest-mini', 'home-unit', 'hub-plus', 'humidifier', 'inkjet-printer', 'kasa-camera-pro', 'kasa-smart-led-lamp', 'qrio-hub', 'sleep', 'smart-power-strip', 'smart-wifi-plug-mini']

✅ All models loaded successfully!


In [8]:
# ═══════════════════════════════════════════════════
# STEP 7B – Prepare test data (clean + adversarial)
# ═══════════════════════════════════════════════════

# Load and preprocess the full dataset
df = load_all_data(DATA_PATH, sample_ratio=SAMPLE_RATIO)
X, y, feature_names = preprocess(df)
del df; gc.collect()

# Encode and scale
# Re-use the SAVED scaler/encoder to ensure consistency with trained models
y_enc = label_encoder.transform(y)
X_mm = scalers['minmax'].transform(X)
X_sc = scalers['standard'].transform(X_mm)
del X; gc.collect()

# Same 75/25 split as training (same random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X_sc, y_enc, test_size=0.25, random_state=42, stratify=y_enc
)
n_features = X_train.shape[1]
print(f'Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}')
print(f'Features: {n_features}')


STEP 1 – Loading IoT IPFIX Home Dataset
  Loading home1_labeled.csv ...
    Shape: (1152574, 37) | Memory: 141.8 MB
  Loading home2_labeled.csv ...
    Shape: (1401331, 37) | Memory: 172.4 MB
  Loading home3_labeled.csv ...
    Shape: (1707785, 37) | Memory: 210.1 MB
  Loading home4_labeled.csv ...
    Shape: (8071108, 37) | Memory: 992.9 MB
  Loading home5_labeled.csv ...
    Shape: (1791772, 37) | Memory: 220.4 MB
  Loading home6_labeled.csv ...
    Shape: (4643641, 37) | Memory: 571.3 MB
  Loading home7_labeled.csv ...
    Shape: (2652647, 37) | Memory: 326.3 MB
  Loading home8_labeled.csv ...
    Shape: (1518573, 37) | Memory: 186.8 MB
  Loading home9_labeled.csv ...
    Shape: (1137142, 37) | Memory: 139.9 MB
  Loading home10_labeled.csv ...
    Shape: (5373492, 37) | Memory: 661.1 MB
  Loading home11_labeled.csv ...
    Shape: (1037093, 37) | Memory: 127.6 MB
  Loading home12_labeled.csv ...
    Shape: (1630017, 37) | Memory: 200.5 MB

Total shape: (32117175, 37)
  After class fi

In [11]:
# ═══════════════════════════════════════════════════
# STEP 7C – Generate adversarial samples from TEST set
# ═══════════════════════════════════════════════════

# Generate adversarial samples from a portion of the test set
N_ADV_TEST = min(5000, len(X_test) // 4)

atk = AdversarialAttackGenerator(feature_names)
atk.fit_centroids(X_train, y_train)

# Take a random subset of test samples for adversarial generation
np.random.seed(42)
adv_indices = np.random.choice(len(X_test), N_ADV_TEST, replace=False)
X_test_for_adv = X_test[adv_indices]
y_test_for_adv = y_test[adv_indices]

print(f'Generating {N_ADV_TEST} adversarial samples from test set...')
X_adv_test = atk.generate_batch(X_test_for_adv, y_test_for_adv)
print(f'✅ Generated {len(X_adv_test)} adversarial test samples')


Generating 5000 adversarial samples from test set...
    Adversarial samples generated: 500/5000
    Adversarial samples generated: 1000/5000
    Adversarial samples generated: 1500/5000
    Adversarial samples generated: 2000/5000
    Adversarial samples generated: 2500/5000
    Adversarial samples generated: 3000/5000
    Adversarial samples generated: 3500/5000
    Adversarial samples generated: 4000/5000
    Adversarial samples generated: 4500/5000
    Adversarial samples generated: 5000/5000
✅ Generated 5000 adversarial test samples


## STEP 7D – CORRECTED Two-Tiered Defense Evaluation

**Key difference from the old code:**
- We create a **MIXED** test set: all clean test samples + all adversarial test samples
- The detector decides for EACH sample whether it's adversarial or not
- Routed accordingly: adversarial → robust model, clean → base model
- Metrics computed on the **full mixed dataset**


In [12]:
# ═══════════════════════════════════════════════════
# STEP 7D – CORRECTED Two-Tiered Defense (Figure 4)
# ═══════════════════════════════════════════════════
#
# Pipeline (Figure 4 du paper):
#   Incoming flow → Model-1 (Detector)
#     → Flagged adversarial → Model-2 (Robust classifier)
#     → Flagged clean       → Model-3 (Base classifier)
#
# CORRECTION: Evaluate on MIXED data (clean + adversarial together)
# This matches the paper's evaluation methodology.

print('=' * 70)
print('CORRECTED Two-Tiered Defense Evaluation (MIXED clean + adversarial)')
print('=' * 70)

# ── Build the MIXED test set ──
# Clean: full test set with true device labels
# Adversarial: adversarial versions of some test samples with their true device labels
X_mixed = np.vstack([X_test, X_adv_test])
y_mixed_true = np.hstack([y_test, y_test_for_adv])  # true device labels for all

# Ground truth for detector: 0=clean, 1=adversarial
y_is_adv = np.hstack([np.zeros(len(X_test)), np.ones(len(X_adv_test))])

print(f'Mixed test set: {len(X_mixed)} samples')
print(f'  Clean:       {len(X_test)} samples')
print(f'  Adversarial: {len(X_adv_test)} samples')

all_results = []

for det_name, detector in detector_models.items():
    print(f'\n{"─" * 60}')
    print(f'Detector: {det_name}')
    print(f'{"─" * 60}')

    # ── Step 1: Detector predicts on the MIXED set ──
    if det_name == 'DNN':
        det_pred = (detector.predict(X_mixed) > 0.5).astype(int).flatten()
    else:
        det_pred = detector.predict(X_mixed)

    # Detector performance
    det_acc = accuracy_score(y_is_adv, det_pred)
    det_pr = precision_score(y_is_adv, det_pred, zero_division=0)
    det_rc = recall_score(y_is_adv, det_pred, zero_division=0)
    det_f1 = f1_score(y_is_adv, det_pred, zero_division=0)
    print(f'  Detector perf: Acc={det_acc:.4f} Pr={det_pr:.4f} Rc={det_rc:.4f} F1={det_f1:.4f}')

    n_flagged_adv = det_pred.sum()
    n_flagged_clean = len(det_pred) - n_flagged_adv
    print(f'  Routing: {int(n_flagged_clean)} → Base (Model-3), {int(n_flagged_adv)} → Robust (Model-2)')

    for cls_name in base_models:
        if cls_name not in robust_models:
            continue
        base_model = base_models[cls_name]
        robust_model = robust_models[cls_name]

        # ── Step 2: Route through pipeline ──
        # For samples detected as clean (0) → use base model (Model-3)
        # For samples detected as adversarial (1) → use robust model (Model-2)
        pred_base = _predict(base_model, X_mixed)
        pred_robust = _predict(robust_model, X_mixed)
        y_pred_defended = np.where(det_pred == 0, pred_base, pred_robust)

        # ── Metrics WITH defense (on mixed set) ──
        acc_def = accuracy_score(y_mixed_true, y_pred_defended)
        pr_def = precision_score(y_mixed_true, y_pred_defended, average='weighted', zero_division=0)
        rc_def = recall_score(y_mixed_true, y_pred_defended, average='weighted', zero_division=0)
        f1_def = f1_score(y_mixed_true, y_pred_defended, average='weighted', zero_division=0)

        # ── Metrics WITHOUT defense (base model on mixed set) ──
        acc_nodef = accuracy_score(y_mixed_true, pred_base)
        pr_nodef = precision_score(y_mixed_true, pred_base, average='weighted', zero_division=0)
        rc_nodef = recall_score(y_mixed_true, pred_base, average='weighted', zero_division=0)
        f1_nodef = f1_score(y_mixed_true, pred_base, average='weighted', zero_division=0)

        # ── Metrics on CLEAN only (base model on clean test) ──
        pred_base_clean = _predict(base_model, X_test)
        f1_clean = f1_score(y_test, pred_base_clean, average='weighted', zero_division=0)
        pr_clean = precision_score(y_test, pred_base_clean, average='weighted', zero_division=0)
        rc_clean = recall_score(y_test, pred_base_clean, average='weighted', zero_division=0)
        acc_clean = accuracy_score(y_test, pred_base_clean)

        # ── Metrics WITHOUT defense on ADV ONLY ──
        pred_base_adv = _predict(base_model, X_adv_test)
        f1_adv_nodef = f1_score(y_test_for_adv, pred_base_adv, average='weighted', zero_division=0)
        pr_adv_nodef = precision_score(y_test_for_adv, pred_base_adv, average='weighted', zero_division=0)
        rc_adv_nodef = recall_score(y_test_for_adv, pred_base_adv, average='weighted', zero_division=0)

        # ── Metrics WITH defense on ADV ONLY ──
        if det_name == 'DNN':
            det_adv_only = (detector.predict(X_adv_test) > 0.5).astype(int).flatten()
        else:
            det_adv_only = detector.predict(X_adv_test)
        pred_base_adv2 = _predict(base_model, X_adv_test)
        pred_robust_adv = _predict(robust_model, X_adv_test)
        y_pred_adv_def = np.where(det_adv_only == 0, pred_base_adv2, pred_robust_adv)
        f1_adv_def = f1_score(y_test_for_adv, y_pred_adv_def, average='weighted', zero_division=0)
        pr_adv_def = precision_score(y_test_for_adv, y_pred_adv_def, average='weighted', zero_division=0)
        rc_adv_def = recall_score(y_test_for_adv, y_pred_adv_def, average='weighted', zero_division=0)

        # Recovery = F1_defended / F1_clean (how much of clean performance is recovered)
        recovery = (f1_adv_def / f1_clean * 100) if f1_clean > 0 else 0

        print(f'\n  Classifier: {cls_name}')
        print(f'    Clean (no attack)        | Pr={pr_clean:.4f} Rc={rc_clean:.4f} F1={f1_clean:.4f}')
        print(f'    Adv WITHOUT defense      | Pr={pr_adv_nodef:.4f} Rc={rc_adv_nodef:.4f} F1={f1_adv_nodef:.4f}')
        print(f'    Adv WITH defense         | Pr={pr_adv_def:.4f} Rc={rc_adv_def:.4f} F1={f1_adv_def:.4f}')
        print(f'    Mixed WITH defense       | Pr={pr_def:.4f} Rc={rc_def:.4f} F1={f1_def:.4f}')
        print(f'    Recovery (F1_adv_def/F1_clean) = {recovery:.1f}%')

        all_results.append({
            'detector': det_name, 'classifier': cls_name,
            'precision_clean': pr_clean, 'recall_clean': rc_clean, 'f1_clean': f1_clean,
            'precision_adv_nodef': pr_adv_nodef, 'recall_adv_nodef': rc_adv_nodef, 'f1_adv_nodef': f1_adv_nodef,
            'precision_adv_defended': pr_adv_def, 'recall_adv_defended': rc_adv_def, 'f1_adv_defended': f1_adv_def,
            'precision_mixed_defended': pr_def, 'recall_mixed_defended': rc_def, 'f1_mixed_defended': f1_def,
            'recovery_pct': recovery,
        })

df_results = pd.DataFrame(all_results)
df_results.to_csv(os.path.join(OUTPUT_PATH, 'step7_corrected_defense_results.csv'), index=False)
print(f'\n✅ Results saved to {OUTPUT_PATH}step7_corrected_defense_results.csv')
df_results


CORRECTED Two-Tiered Defense Evaluation (MIXED clean + adversarial)
Mixed test set: 661252 samples
  Clean:       656252 samples
  Adversarial: 5000 samples

────────────────────────────────────────────────────────────
Detector: RF
────────────────────────────────────────────────────────────
  Detector perf: Acc=0.9999 Pr=0.9874 Rc=1.0000 F1=0.9936
  Routing: 656188 → Base (Model-3), 5064 → Robust (Model-2)


NameError: name '_predict' is not defined

In [ ]:
# ═══════════════════════════════════════════════════
# CORRECTED Figure 10: With vs Without Defense
# ═══════════════════════════════════════════════════
# Use the BEST detector (highest F1) for plotting

# Find best detector
det_f1s = {}
for det_name in detector_models:
    if det_name == 'DNN':
        dp = (detector_models[det_name].predict(X_mixed) > 0.5).astype(int).flatten()
    else:
        dp = detector_models[det_name].predict(X_mixed)
    det_f1s[det_name] = f1_score(y_is_adv, dp, zero_division=0)
best_det = max(det_f1s, key=det_f1s.get)
print(f'Best detector: {best_det} (F1={det_f1s[best_det]:.4f})')

# Filter results for best detector
best_res = df_results[df_results['detector'] == best_det].copy()
models_list = best_res['classifier'].tolist()

# ── Figure 10 style: grouped bar chart ──
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: Without Defense (Precision, Recall, F1 on adversarial only)
ax = axes[0]
x = np.arange(len(models_list))
width = 0.25
colors_nodef = ['#4472C4', '#ED7D31', '#A5A5A5', '#FFC000']

bars1 = ax.bar(x - width, best_res['precision_adv_nodef'] * 100, width, label='Precision', color='#4472C4')
bars2 = ax.bar(x, best_res['recall_adv_nodef'] * 100, width, label='Recall', color='#ED7D31')
bars3 = ax.bar(x + width, best_res['f1_adv_nodef'] * 100, width, label='F1-Score', color='#A5A5A5')

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2., h + 1, f'{h:.1f}%',
                    ha='center', va='bottom', fontsize=9, rotation=0)

ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Without Defence', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models_list, fontsize=11)
ax.legend(fontsize=10)
ax.set_ylim(0, 110)
ax.grid(axis='y', alpha=0.3)

# Right: With Defense (Precision, Recall, F1 on adversarial with defense)
ax = axes[1]
bars1 = ax.bar(x - width, best_res['precision_adv_defended'] * 100, width, label='Precision', color='#4472C4')
bars2 = ax.bar(x, best_res['recall_adv_defended'] * 100, width, label='Recall', color='#ED7D31')
bars3 = ax.bar(x + width, best_res['f1_adv_defended'] * 100, width, label='F1-Score', color='#A5A5A5')

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2., h + 1, f'{h:.1f}%',
                    ha='center', va='bottom', fontsize=9, rotation=0)

ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('With Defence', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models_list, fontsize=11)
ax.legend(fontsize=10)
ax.set_ylim(0, 110)
ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'Figure 10: Performance With Robustness Measures (Detector: {best_det})',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'Figure_10_CORRECTED_Defense.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 10 (Corrected) saved')


In [ ]:
# ═══════════════════════════════════════════════════
# Summary: Compare all detectors × classifiers
# ═══════════════════════════════════════════════════

print('\n' + '=' * 80)
print('SUMMARY: Two-Tiered Defense Results (All Detectors × All Classifiers)')
print('=' * 80)
print(f'{"Detector":>10} | {"Classifier":>10} | {"F1 Clean":>10} | {"F1 Adv NoD":>12} | {"F1 Adv Def":>12} | {"Recovery":>10}')
print('-' * 80)
for _, row in df_results.iterrows():
    print(f'{row["detector"]:>10} | {row["classifier"]:>10} | '
          f'{row["f1_clean"]*100:>9.1f}% | '
          f'{row["f1_adv_nodef"]*100:>11.1f}% | '
          f'{row["f1_adv_defended"]*100:>11.1f}% | '
          f'{row["recovery_pct"]:>9.1f}%')
print('=' * 80)
